# Mountain NER Dataset Creation

This notebook demonstrates the complete dataset creation pipeline used to build the training data for the Mountain Named Entity Recognition (NER) task.

The dataset was generated automatically using real mountain names collected from the Open Peaks project. The resulting dataset was later used to fine-tune a BERT-based NER model.

## Dataset Creation Pipeline

The dataset generation consists of three main stages:

1. Collect real mountain names from the Open Peaks dataset.
2. Generate positive BIO-annotated training examples.
3. Generate negative examples that do not contain mountain entities.

The overall pipeline is shown below:

```
Open Peaks
      │
      ▼
download_mountains.py
      │
      ▼
mountains.csv
      │
      ▼
generate_dataset.py
      │
      ▼
ner_dataset.json
```

## Step 1. Collect Mountain Names

Instead of creating mountain names manually, a publicly available dataset from the Open Peaks project was used.

The `download_mountains.py` script downloads the mountain index, extracts mountain names, removes duplicates, and stores the result as a CSV file.

Using real geographical names improves the realism of the generated dataset and reduces the chance of introducing artificial entities.

In [4]:
import pandas as pd

mountains = pd.read_csv("../data/raw/mountains.csv")

mountains.head(10)

,mountain
0,A'Mhaighdean
1,Abbott Butte
2,Acatenango
3,Achen Niyeu
4,Acho
5,Ackerlspitze
6,Aconcagua
7,Acotango
8,Adam's Peak
9,Adi Kailash


In [2]:
print(f"Total mountain names: {len(mountains)}")

Total mountain names: 2924


## Step 2. Generate Positive Training Examples

Each mountain name is inserted into several sentence templates.

For every generated sentence, BIO labels are created automatically.

Example:

Sentence:

```
We successfully climbed Mount Elbrus during the expedition.
```

BIO labels:

```
We           O
successfully O
climbed      O
Mount        B-MOUNTAIN
Elbrus       I-MOUNTAIN
during       O
the          O
expedition   O
```

This approach allows the dataset to be generated automatically while preserving correct BIO annotations.

## Step 3. Generate Negative Examples

To reduce false positive predictions, additional negative examples were generated.

These sentences contain entities such as:

- cities
- people
- companies
- organizations
- technologies

Since no mountain names are present, every token receives the label **O**.

Including negative examples makes the model more robust and helps prevent incorrect mountain predictions.

In [5]:
import json

with open("../data/processed/ner_dataset.json", encoding="utf-8") as f:
    dataset = json.load(f)

len(dataset)

7000

## Dataset Format

Each sample is stored as a JSON object containing:

- unique sample ID;
- tokenized sentence;
- BIO labels.

In [8]:
dataset[1]

{'id': 1,
 'sentence': 'The trail leading to Pollock Mountain is considered challenging.',
 'tokens': ['The',
  'trail',
  'leading',
  'to',
  'Pollock',
  'Mountain',
  'is',
  'considered',
  'challenging',
  '.'],
 'ner_tags': [0, 0, 0, 0, 1, 2, 0, 0, 0, 0]}